# 3. App Registrations and Managed Identities

AZ-500 expects you to configure OAuth flows, manage consent, and implement managed identities.
This builds on the [Azure Authentication lab](../../../08-enterprise/azure-authentication/).

## Before you run this notebook

1. Start the tiny mock Entra ID (built from the `08-enterprise/azure-authentication` lab):
   ```bash
   cd security-certs/az-500/01-identity-and-access
   docker compose up -d
   uv sync
   ```
2. In VS Code, pick the `.venv` kernel at the top-right. Reload the window if it's not listed.
3. Check the mock is reachable:
   ```bash
   curl http://localhost:9100/health
   ```
   You should see `{"status":"ok",...}`.

> **Heads-up:** The mock speaks just enough OAuth2 for this lab (`client_credentials`,
> `password` / ROPC, and on-behalf-of). Real Entra ID has many more features and you'd
> almost never enable ROPC in production.

## App registration checklist

| Setting | Where in the portal | Purpose |
|---------|---------------------|---------|
| **Display name** | Overview | Human-readable identifier |
| **Supported account types** | Authentication | Single tenant, multi-tenant, or personal |
| **Redirect URIs** | Authentication | Where tokens are sent after interactive sign-in |
| **Client secret or certificate** | Certificates & secrets | Prove identity for confidential clients |
| **API permissions** | API permissions | What resources the app can access |
| **Expose an API** | Expose an API | Scopes other apps can request |
| **App roles** | App roles | Application-level permissions (no user) |


In [ ]:
import httpx, json, base64

ENTRA = 'http://localhost:9100/contoso'
TOKEN_URL = f'{ENTRA}/oauth2/v2.0/token'

def decode(token: str) -> dict:
    # Quick-and-dirty JWT payload decoder. DO NOT use this to validate tokens in
    # real code. Your API library (FastAPI + python-jose, MSAL, etc.) must verify
    # the signature, issuer, audience and expiry before trusting anything.
    payload = token.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(payload + '=' * (-len(payload) % 4)))

# --- 1. Delegated permission (user is present) ------------------------------
# The api-a application is acting ON BEHALF OF Alice. The token has an `scp`
# claim listing the delegated scopes she's consented to.
print('=== 1. Delegated permission (OAuth scope) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'password',
    'client_id': 'api-a-client-id',
    'client_secret': 'api-a-secret-value',
    'username': 'alice@contoso.com',
    'password': 'alice-password',
    'scope': 'api://api-b/Files.Read',
})
r.raise_for_status()
delegated = decode(r.json()['access_token'])
print('Has "scp" claim (delegated), no "roles" claim.')
print(f'  scp (scopes): {delegated.get("scp")}')
print(f'  upn (user):   {delegated.get("upn")}')
print(f'  aud:          {delegated.get("aud")}')

# --- 2. Application permission (no user) ------------------------------------
# A daemon/cron calling api-b on its own behalf. The token has a `roles` claim
# listing the app roles the daemon was pre-assigned. There is no user.
print('\n=== 2. Application permission (app role) ===')
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
r.raise_for_status()
app_only = decode(r.json()['access_token'])
print('Has "roles" claim (app-only), no user claims.')
print(f'  roles:        {app_only.get("roles")}')
print(f'  upn:          {app_only.get("upn", "(none - app-only token)")}')
print(f'  aud:          {app_only.get("aud")}')


## Bad to Best: handling the daemon's credentials

The daemon above authenticates with a **client secret** we pasted into code. That's exactly the anti-pattern managed identities exist to fix.


In [ ]:
approaches = [
    ('BAD',
     'Client secret hard-coded in source control.',
     'One `git clone` and an attacker has your app\'s identity. Rotation means a code change.'),
    ('OKAY',
     'Client secret in a Key Vault, fetched at startup.',
     'Better, but *something* still has to authenticate to Key Vault without a secret. Chicken and egg.'),
    ('GOOD',
     'Client certificate (private key in Key Vault, rotated automatically).',
     'Stronger than secrets, but you still manage rotation.'),
    ('BEST',
     'Managed identity (system- or user-assigned) on the hosting resource.',
     'Azure handles creation, rotation and retirement. There is no secret to leak.'),
    ('BEST (for CI/CD)',
     'Workload Identity Federation (OIDC) - e.g. GitHub Actions trusts Entra ID directly.',
     'No secret to rotate, no managed identity needed outside Azure.'),
]
for tag, setup, impact in approaches:
    print(f'{tag:<18}  {setup}')
    print(f'                    -> {impact}\n')


## OAuth consent

| Type | Who approves | When |
|------|-------------|------|
| **User consent** | The individual user | Low-risk delegated permissions (e.g. `User.Read`) |
| **Admin consent** | A tenant admin | High-risk delegated, or *any* application permission |
| **Pre-authorized** | App owner | First-party apps, or verified publishers |

### Managing consent

```bash
# Grant admin consent for an app.
az ad app permission admin-consent --id <app-id>

# See what's currently granted.
az ad app permission list --id <app-id>

# Restrict user consent to verified publishers and low-risk permissions.
az rest --method PATCH \
  --uri 'https://graph.microsoft.com/v1.0/policies/authorizationPolicy' \
  --body '{"defaultUserRolePermissions": {"permissionGrantPoliciesAssigned": ["managePermissionGrantsForSelf.microsoft-user-default-low"]}}'
```

### Exam tip: illicit consent grants

A common attack: an attacker tricks a user into consenting to an app that asks for `Mail.Read` or `Files.ReadWrite.All`. The attacker doesn't need the user's password - consent is enough. Defend by:

1. Restricting user consent to verified publishers.
2. Requiring admin consent for high-risk permissions.
3. Running the **Admin consent workflow** so users *request* consent instead of blindly approving.
4. Reviewing existing grants regularly.

---
## Managed identities - when to use which

| | System-assigned | User-assigned |
|-|----------------|---------------|
| **Lifecycle** | Created/deleted with the resource | Independent Azure resource |
| **Sharing** | One resource only | Can attach to many resources |
| **Use case** | Single-purpose workload | Shared identity (e.g. many VMs hitting the same Key Vault) |
| **Deployment slots** | New identity per slot | Same identity across slots |

### Azure CLI reference


In [ ]:
cli = {
    'Enable system-assigned MI on a VM':
        'az vm identity assign -g rg-prod -n my-vm',
    'Create a user-assigned MI':
        'az identity create -g rg-prod -n my-app-identity',
    'Attach user-assigned MI to a Container App':
        'az containerapp identity assign -g rg-prod -n my-app '
        '--user-assigned /subscriptions/.../my-app-identity',
    'Grant MI access to Key Vault secrets':
        'az role assignment create --assignee <MI-principal-id> '
        '--role "Key Vault Secrets User" --scope <kv-resource-id>',
    'Grant MI access to Storage blobs':
        'az role assignment create --assignee <MI-principal-id> '
        '--role "Storage Blob Data Contributor" --scope <storage-resource-id>',
    'Fetch a token from inside a VM (IMDS endpoint)':
        'curl "http://169.254.169.254/metadata/identity/oauth2/token'
        '?api-version=2018-02-01&resource=https://vault.azure.net" -H Metadata:true',
}
for desc, cmd in cli.items():
    print(f'# {desc}')
    print(f'{cmd}\n')


### IMDS - the Instance Metadata Service

Inside Azure VMs, App Services, Functions and Container Apps, managed identities ask **IMDS** at `169.254.169.254` for tokens. The Azure SDK calls this automatically via `DefaultAzureCredential`:

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# Same code works on your laptop (az login), in CI (env vars / federated creds),
# and in Azure (managed identity via IMDS).
credential = DefaultAzureCredential()
client = SecretClient(vault_url='https://my-kv.vault.azure.net', credential=credential)
db_password = client.get_secret('db-password').value
```

## Workload Identity Federation (federated credentials)

For workloads *outside* Azure (GitHub Actions, AKS pods, GitLab, other clouds) you can
still get an Entra ID token **without** any secret, using **federated credentials**.
The external platform issues its own OIDC token; Entra ID trusts it based on `issuer`,
`subject` and `audience`. Zero long-lived secrets, full audit trail.

```yaml
# Minimal GitHub Actions step that logs in to Azure using federation.
- uses: azure/login@v2
  with:
    client-id:       ${{ secrets.AZURE_CLIENT_ID }}
    tenant-id:       ${{ secrets.AZURE_TENANT_ID }}
    subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}
    # NOTE: no client-secret! Federation uses the OIDC token GitHub mints for this job.
```

```bash
# On Azure: add a federated credential to the app registration.
az ad app federated-credential create --id <app-id> --parameters '{
  "name": "github-main",
  "issuer": "https://token.actions.githubusercontent.com",
  "subject": "repo:my-org/my-repo:ref:refs/heads/main",
  "audiences": ["api://AzureADTokenExchange"]
}'
```

### Exam tips

- Always prefer **managed identity** over stored secrets.
- For AKS: use **workload identity** (federated, via OIDC) instead of pod-mounted secrets.
- For GitHub Actions / other CI: use **federated credentials** instead of a stored client secret.
- `DefaultAzureCredential` is the recommended credential class for all scenarios.

---
## Summary

| Concept | Key point |
|---------|-----------|
| **App registration** | Client ID + (secret / cert / federated cred), API permissions, exposed scopes, app roles |
| **Delegated vs application** | `scp` claim (user present) vs `roles` claim (app-only) |
| **Consent management** | Restrict user consent, require admin consent for risky permissions |
| **System-assigned MI** | 1:1 with its resource, auto-deleted |
| **User-assigned MI** | Reusable across resources, lifecycle you manage |
| **Workload Identity Federation** | Secretless auth for CI/CD, AKS, other clouds |
| **DefaultAzureCredential** | One credential class that works everywhere |

**Next lab**: [02 - Secure Networking](../../02-networking/)
